# XLM-RoBERTa Language Detection

**Model:** papluca/xlm-roberta-base-language-detection | **Size:** 278MB | **Product:** prod-rwkhoi4b6uku4

A fine-tuned XLM-RoBERTa model for automatic language identification across 20 languages. Built on the multilingual XLM-RoBERTa base, it achieves high accuracy on diverse text inputs and is ideal for routing and preprocessing pipelines.

## Use Cases
- Automatic language routing in multilingual customer support
- Content classification and moderation pipelines
- Preprocessing for machine translation workflows
- Multilingual data labeling and quality filtering

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'xlm-roberta-language-detect'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send text samples in various languages. The model returns the detected language label and confidence score.

In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=region)

# Sample texts in different languages
texts = [
    'The quick brown fox jumps over the lazy dog.',       # English
    'Le renard brun rapide saute par-dessus le chien paresseux.',  # French
    'Der schnelle braune Fuchs springt uber den faulen Hund.',     # German
]

for text in texts:
    payload = json.dumps({'inputs': text})
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        result_raw = response['Body'].read().decode('utf-8')
        try:
            result = json.loads(result_raw)
            print(f'Text: "{text[:50]}..." -> Language: {result}')
        except json.JSONDecodeError:
            print(f'Raw response for "{text[:30]}": {result_raw}')
    except Exception as e:
        print(f'Inference failed for "{text[:30]}": {e}')
        raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')